### 1.1 Tensor creation and inspection
In this section, you will learn how to create and manipulate PyTorch tensors, exploring their attributes and behavior through simple examples. Start by importing the torch library and creating a few small tensors manually to understand their structure and key properties:

- **Creating tensors**: Define a one-dimensional tensor containing a few floating-point values. Then, create a second tensor of the same shape and perform a simple element-wise operation (for example addition or multiplication). Print the resulting tensor and verify its contents.

- **Inspecting tensor attributes**: Examine the tensor’s shape, number of dimensions, and data type using the appropriate attributes. Check which precision is used by default and compare it to NumPy’s behavior.

- **Changing tensor data types**: Experiment with converting a tensor to another data type using either the .to() method or the shorthand functions (for example, .float() or .double()). The .to() method is a versatile function that allows you to move tensors between devices (CPU and GPU) or change their properties, such as data type and precision, by passing the desired torch.dtype as an argument. For example, you can use tensor.to(torch.float64) to convert a tensor to double precision, or tensor.to("cuda") to move it to the GPU. After applying the conversion, verify that the tensor’s dtype changes accordingly.

- Performance comparison of data types: Using the torch.randn function, create two large random matrices (for example, of size 1000 × 1000) and measure the time required to perform a matrix multiplication in both float32 and float64. After these analyses, why do you think PyTorch uses float32 as the default type instead of float64?

Info: You can use the timeit function by repeatedly running the matrix multiplication (for example,
100 times) and comparing the total times obtained for the two data types.
```python
from timeit import timeit
# Example timing for float64 and float32 matrix multiplication
t64 = timeit(lambda: M1_64 @ M2_64, number=100)
t32 = timeit (lambda: M1_32 @ M2_32, number =100)
print(f"Time for matrix multiplication (float64): {t64:.4f}s")
print(f"Time for matrix multiplication (float32): {t32:.4f}s")
```

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

from sklearn.model_selection import train_test_split

import numpy as np

import matplotlib.pyplot as plt


In [ ]:
# cerate a tensor with floating values
x = torch.tensor([
    [1,2,3],
    [4,5,6],
    [7,8,9]
    ])
print(x)
print(x.shape)

In [ ]:
# create a second tensor
y = torch.tensor([
    [9,8,7],
    [6,5,4],
    [3,2,1]
    ])

z = torch.tensor([
    [1]
])

z_1d = torch.tensor([1])

# addition element wise
print(f'Sum element wise, between X: \n{x} \nand Y: \n{y} \nis: \n{x+y}\n')

print('--------------------------')

print(f"Broadcasting between X: \n{x}\n{x.shape}\n\nand Z: \n\n{z}\n{z.shape}\n\nis: \n\n{x+z}\n{(x+z).shape}\n")  # tensor allow broadcasting ... yuppy ... :(

print('--------------------------')

print(f"Broadcasting between X: \n{x}\n{x.shape}\n\nand Z':\n\n{z_1d}\n{z_1d.shape}\n\nis:\n\n{x+z_1d}\n{(x+z_1d).shape}\n")   # ok it performs broadcasting both if the tensor is 1D or 2D



In [ ]:
# default torch precision
print('TORCH:')
print(torch.get_default_dtype())

# float32
# or int64 if all elements in the tensor are int

tensor = torch.tensor(3)
tensor_2 = torch.tensor([1,2,3,4,5])
tensor_3 = torch.tensor([1,2,3,4,5.12])
print(tensor.dtype)
print(tensor_2.dtype)
print(tensor_3.dtype)

print('------------------------------')
print('NUMPY:')
array = np.array(3)
array_2 = np.array([1,2,3,4,5])
array_3 = np.array([1,2,3,4,5.12])
print(array.dtype)
print(array_2.dtype)
print(array_3.dtype)

# float64
# or int64 if all elements in the array are int

In [ ]:
# convert tensor to another data type with '.to()'
# tensor.to(torch.float64)
# tensor.to('cuda') --> I don't have a NVIDIA GPU so use 'mps'

# from int64 to float64
tensor_2 = torch.tensor([1,2,3,4,5])

print(tensor_2.dtype)
tensor_2_float64 = tensor_2.to(torch.float64)
print(tensor_2_float64.dtype)

# move it from CPU to GPU
tensor_2_gpu = tensor_2.to('mps')
# tensor_2_float64_gpu = tensor_2_float64.to('mps') --> YOU CANNOT, ONLY FOR FLOAT32 OR INT64

tensor_2_float32 = tensor_2_float64.to(torch.float32)
tensor_2_float32_gpu = tensor_2_float32.to('mps')
print(tensor_2_gpu.dtype)
print(tensor_2_float32_gpu.dtype)

In [ ]:
# compare performance float32 and float64
from timeit import timeit

tensor_1_32 = torch.randn(100, 100)
tensor_2_64 = tensor_1_32.to(torch.float64)
print(tensor_1_32.dtype)
print(tensor_2_64.dtype)

# Example timing for float64 and float32 matrix multiplication
t64 = timeit(lambda: tensor_2_64 @ tensor_2_64, number=100)
t32 = timeit (lambda: tensor_1_32 @ tensor_1_32, number =100)
print(f"Time for matrix multiplication (float64): {t64:.4f}s")
print(f"Time for matrix multiplication (float32): {t32:.4f}s")

# float64 take 4 times more computational time

---

### 2. Datasets and Dataloaders
Usually, we don’t work with “just one tensor”. In most cases, we have many samples consisting of inputs and corresponding labels, and we need a simple way to access them for training. PyTorch provides this through Dataset, which enables easy handling of datasets. Once the dataset is created, we wrap it into a **DataLoader**, which provides an easy and efficient way to iterate over the data in batches. The DataLoader automatically groups samples, handles shuffling, and prepares batches ready to be fed to the model during training.

In this first part, instead of defining a custom dataset class, we will use the built-in **TensorDataset**, which conveniently pairs input and target tensors so that each element of the dataset directly corresponds to a sample (X,y).
> Your goal is to generate a one-dimensional regression problem and then use a DataLoader to iterate over it:
> 	- Input: each sample has one feature → a single number x (not 4 features, not an image, just one scalar).
> 	- Target: each sample has one output y, also a scalar.

- Dataset generation: Create a dataset containing n= 2048 samples. Use the torch.randn function to generate a one-dimensional input tensor X of shape (n,1), representing normally distributed random features. Then, define the target tensor y as a linear transformation of X with added Gaussian noise according to the relation:

Y = 5X + 3 + $\epsilon$

where $\epsilon$ is a small random noise term sampled from a normal distribution. Verify that both X and y
have the expected shapes and compatible data types.

- Creating a Dataset object: Wrap the tensors X and y into a TensorDataset object. This class is a convenient way to combine multiple tensors into a dataset where each element corresponds to a pair (Xi,yi). Check that indexing the dataset (for example, dataset[0]) returns a tuple containing one sample and its label.

- Building a DataLoader and inspect batches: Create a DataLoader from your dataset to enable efficient iteration in batches. Set a batch size of 256 samples and enable shuffling (shuffle=True). Iterate once over your DataLoader and print the shapes of the input and target batches. Verify that the shapes are consistent with the chosen batch size (for example, [256, 1] for both X and y). Then, change the batch_size parameter and observe how the shapes of the resulting batches change.

In [ ]:
# Build the dataset

# number of rows / samples. The dataset will have n rows and 1 column
n = 2048
x = torch.randn(n, 1)


# y are the target values and are the following transformation of x: y = 5x + 3 + noise (?)
y = (5*x) + 3
print(x)
print(x.shape)

print('----------------------------')

print(y)
print(y.shape)

In [ ]:
# not requested, but do it
# GODDAMN SKLEARN IS GOATED, IT WORKS ON TENSORS TOO!!
# AND RETURNS TENSORS RAAAAHHHHH
x_train, x_eval, y_train, y_eval = train_test_split(x,y,test_size=0.2, random_state=42)

sanity = False
if sanity:
    print(x_train)
    print(x_train.dtype)
    print(x_eval)
    print(x_eval.dtype)

# create TensorDataset object and pass data and corresponding labels
train_dataset = TensorDataset(x_train, y_train)
print(train_dataset[0])

# it basically creates couples, the couple x_i and y_i
# (tensor(0.2978), tensor(4.4891))
# (x_i , y_i)

In [ ]:
train_dataset = TensorDataset(x_train, y_train)
# create DataLoader object to create batches
    # pass him the TensorDataset, which is the original DS splitted into couples (x_i, y_i)
    # pass him how many batches it has to create --> it glues together 256 couples together
    # shuffle = True, it mixes them
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

# at each iteration the batch is 256 rows of x and 256 rows of y
# keep going until there are no more rows
for x_batch, y_batch in train_loader:
    print(f'x_batch shape: {x_batch.shape}')
    print(f'y_batch shape: {y_batch.shape}')
    print()
    

---

### 3 Building and understanding a simple linear model
In this section, you will implement and analyze a simple linear model in PyTorch to train it on the dataset you have just created. A univariate linear model is one of the simplest forms of a machine learning model and can be defined by the equation:

$y = wx + b$

where x is the input feature, wis the weight, bis the bias, and y is the predicted output.

##### 3.1 Model definition: SimpleLinearModel
The SimpleLinearModel class is a minimal example of a learnable model implemented using PyTorch’s module interface. It extends the base class nn.Module, which is used to define all neural networks in PyTorch. The model consists of a single linear layer that performs the univariate linear transformation.

- Create the class structure: Define a class SimpleLinearModel inheriting from nn.Module. Inside the constructor (__init__),create a single linear layer using nn.Linear(input_size, output_size). In this exercise, both the input and output sizes are equal to 1, meaning that the model will learn to map one scalar input to a single scalar output.

- Define the forward pass: Implement the forward() method to define how the input data flows through the layer. This method should take an input tensor x and return the result of applying the linear transformation to it. This defines how PyTorch will compute the model’s prediction during both training and inference.

- Initialize and analyze your model: Once the class is defined, create an instance of the model and test it with a single input value. Check that the model returns an output tensor of the correct shape. Inspect also the parameters of the model (the weight and bias) and note that they are randomly initialized. These parameters have the attribute requires_grad=True, meaning that PyTorch will automatically track their gradients during the backward pass.
```python
class SimpleLinearModel (nn.Module):
    def __init__(self, input_size, output_size):
        super (SimpleLinearModel, self).__init__()
        self.linear=……・
    
    def forward (self, x):
        return...

# Instantiate the model
model=…..
```

In [ ]:
class SimpleLinearModel(nn.Module):
    
    def __init__(self):
        super (SimpleLinearModel, self).__init__()
        
        # define all the layers
        self.layer_1 = nn.Linear(1,1)
    
    def forward (self, x):
        x = self.layer_1(x)
        return x










# Instantiate the model
model = SimpleLinearModel()
for x_batch, y_batch in train_loader:
    logits = model(x_batch)

    verbose = False
    if verbose:
        print('What I fed the model with:')
        print('Data:')
        print(x_batch)
        print(x_batch.shape)
        print('Respective labels:')
        print(y_batch)
        print(y_batch.shape)
        print()
        print('What the model returned:')
        print(logits.shape)
        print(logits)

        alternative_1 = False
        if alternative_1:
            print('The parameters are:')
            print("\tWeight:", model.layer_1.weight)
            print("\tBias:", model.layer_1.bias)

        # OR

        alternative_2 = True
        if alternative_2:
            print("=== Model parameters ===")
            for name, param in model.named_parameters():
                print(name, param.shape)
                print(param)

    break


*=== Model parameters ===*  
*layer_1.weight torch.Size([1, 1])*  
*Parameter containing:*  
*tensor([[-0.5745]], requires_grad=True)*  
*layer_1.bias torch.Size([1])*  
*Parameter containing:*  
*tensor([-0.3481], requires_grad=True)*  

The model is currently learning the function:  

$y = weight * x + bias$

with weight = -0.5745 and bias = -0.3481 (for now).

#### layer_1.weight torch.Size([1, 1])
- layer_1 is nn.Linear(1, 1).
- For nn.Linear(in_features=1, out_features=1):
- the weight matrix has shape (out_features, in_features) = (1, 1).

So: *tensor([[-0.5745]])*  
means the weight matrix is:  

$W = \begin{bmatrix} -0.5745 \end{bmatrix}$

Since there’s only 1 input and 1 output, it’s just a single scalar weight.

#### layer_1.bias torch.Size([1])
- The bias has shape (out_features,) = (1,).  

So: *tensor([-0.3481])*  
means your bias vector is:
$b = \begin{bmatrix} -0.3481 \end{bmatrix}$

##### SO:
For each input sample x the layer computes:

$y = x \cdot W^T + b$

Here, since everything is 1D:

$y = (-0.5745) \cdot x + (-0.3481)$

#### requires_grad=True
Both weight and bias say:  
*requires_grad=True*  
- PyTorch will compute gradients for these parameters during .backward().
- They will be updated by the optimizer (optimizer.step()).

---

##### 3.2 Criterion and optimizer
During training, two components control the learning process: the criterion and the optimizer.
- Criterion (loss function): The loss function measures the discrepancy between the model’s predictions and the true labels. In this exercise, since the goal is to predict a continuous value, we will use the Mean Squared Error (MSE) loss.

- Optimizer: The optimizer updates the model’s parameters using the computed gradients. For this exercise, we will use Stochastic Gradient Descent (SGD) with a learning rate of 0.01. The optimizer adjusts the weights and bias in the direction that minimizes the loss.


##### 3.3 Training loop
The training loop is the process through which the model learns from data. It involves multiple epochs, where each epoch corresponds to one full pass over the dataset. For each batch, the following sequence of steps occurs:
- Forward pass: Feed inputs through the model to obtain predictions.
- Loss computation: Compare predictions to true labels using the loss function.
- Backward pass: Compute gradients of the loss with respect to model parameters.
- Parameter update: Use the optimizer to update weights and biases.
- Gradient reset: Clear old gradients using optimizer.zero_grad() to prevent accumulation.

```python
num_epochs = 50
losses, weights, biases = [], [], []
model.train ()
for epoch in range (num_epochs):
    running-loss = 0.0
    for inputs, labels in trainloader:
        inputs, labels = ...
        optimizer.zero_grad ()

        # Forward pass
        outputs = ...
        1oss=・・・
        
        # Backward pass and optimization
        •••
        
        # Track values
        losses. append (loss.item())
        weights.append(model.layer_1.weight.item())
        biases. append (model.layer_1.bias.item())
        running-loss += loss. item ()
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {running_loss/len( trainloader):.4f}")
```

# Hold on 'model.train()'???
This is not “train the model now”.
It’s: “put the model in training mode”.

PyTorch models have two important modes:
> - **model.train()** → training mode
> - **model.eval()** → evaluation/inference mode

> “we are in training phase; behave like a training model”.

In [ ]:
# create an object belonging to the SimpleLinearModel class + build optimizer + build loss function
    # use the Mean Squared Error (MSE) loss
    # use Stochastic Gradient Descent (SGD) with lr = 0.01

model = SimpleLinearModel()

loss_fn = nn.MSELoss()

optimizer = optim.SGD(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 50
losses, weights, biases = [], [], []

model.train () # set training mode

for epoch in range(num_epochs):
    running_loss = 0.0
    
    for x_batch, y_batch in train_loader:
        # gradient reset
        optimizer.zero_grad()

        # Forward pass
        logits = model(x_batch)

        # loss computation
        loss = loss_fn(y_batch, logits)
        
        # Backward pass and optimization
        loss.backward()

        optimizer.step()
        
        # Track values
        losses.append(loss.item())
        weights.append(model.layer_1.weight.item())
        biases.append(model.layer_1.bias.item())
        running_loss += loss.item()
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {running_loss/len(train_loader):.4f}")

> After training, plot the evolution of the loss, weight, and bias to visualize the learning process.

In [ ]:
ys = [losses, weights, biases]
x = np.linspace(0, num_epochs, len(losses))
# inizia da 0, arriva a num_epochs (es. 50) ed usa esattamente len(losses) punti
# --> x ha la stessa lunghezza di losses.

plt.figure(figsize=(8, 10))

# 1) Loss
plt.subplot(3, 1, 1)
plt.plot(x, losses)
plt.ylabel("Loss")

# 2) Weight
plt.subplot(3, 1, 2)
plt.plot(x, weights)
plt.ylabel("Weight")

# 3) Bias
plt.subplot(3, 1, 3)
plt.plot(x, biases)
plt.ylabel("Bias")
plt.xlabel("Training step")  # o "Epoch" se logghi una volta per epoch

plt.tight_layout()
plt.show()